In [3]:
import subprocess, shutil, copy
from TprParser.TprReader import TprReader

In [4]:
def Pressure(fname):
    reader = TprReader(fname)
    ref_p = [
        100, 0, 0,
        0, 100, 0,
        0, 0, 100
    ]
    compress = [
        4.5E-5, 0, 0,
        0, 4.5E-5, 0,
        0, 0, 4.5E-5
    ]
    assert len(ref_p) == 9
    assert len(compress) == 9
    reader.set_pressure('No', 'Isotropic', 1.0, ref_p, compress)


In [5]:
def run_cmd(cmd:str):
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode != 0:
        raise Exception('\nError occurred from command: \n\t%s!!!' %cmd)
    
def MD(inittpr:str, nsteps:int = 10):
    reader = TprReader(inittpr)
    coords = reader.get_xvf('X') # get coords from tpr
    natmA = 120
    natmB = 132
    natm = natmA+natmB

    assert natm == coords.shape[0]
    for i in range(nsteps):
        # move two molecules distance of Z axis each 2.0 A
        tempcoords = copy.deepcopy(coords)
        tempcoords[:natmA,     2] += 0.05 * i
        tempcoords[natmA:natm, 2] -= 0.05 * i
        reader.set_xvf('X', tempcoords)
        # rename new.tpr to em_{i}.tpr
        suffix = inittpr.split(".tpr")[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')
    print("Finished!")

In [6]:
def Temperature(fname:str, nsteps:int=2):
    tau_t = [1.0, 1.0] # coupling constant
    reader = TprReader(fname)
    for i in range(nsteps):
        ref_t = [100+i*100, 100+i*100] # 100, 200, 300 K
        reader.set_temperature("Vrescale", tau_t, ref_t)
        suffix = fname.split('.tpr')[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')


In [7]:
def MDP_Integer(fname, key, val):
    reader = TprReader(fname)
    reader.set_mdp_integer(key, val)

In [8]:
def get_xvf(fname, type):
    reader = TprReader(fname)
    return reader.get_xvf(type)

In [9]:
if __name__ == '__main__':
    arr = get_xvf('em.tpr', 'x')
    print(arr)
    # MD('em.tpr', 30)

[[2.471 2.521 5.438]
 [2.474 2.613 5.478]
 [2.468 2.528 5.339]
 [2.39  2.473 5.471]
 [2.591 2.447 5.478]
 [2.59  2.435 5.577]
 [2.59  2.304 5.419]
 [2.501 2.266 5.441]
 [2.696 2.214 5.482]
 [2.693 2.123 5.442]
 [2.787 2.254 5.467]
 [2.679 2.207 5.58 ]
 [2.605 2.302 5.266]
 [2.604 2.207 5.234]
 [2.529 2.352 5.225]
 [2.691 2.346 5.24 ]
 [2.712 2.533 5.435]
 [2.701 2.641 5.379]
 [2.837 2.489 5.459]
 [2.85  2.401 5.504]
 [2.951 2.57  5.418]
 [2.948 2.581 5.319]
 [2.942 2.713 5.477]
 [2.851 2.745 5.456]
 [3.043 2.809 5.413]
 [3.033 2.899 5.454]
 [3.136 2.774 5.427]
 [3.025 2.815 5.315]
 [2.958 2.715 5.63 ]
 [2.952 2.81  5.662]
 [2.885 2.661 5.671]
 [3.047 2.677 5.655]
 [3.078 2.492 5.46 ]
 [3.073 2.382 5.517]
 [3.199 2.543 5.435]
 [3.207 2.631 5.39 ]
 [3.318 2.469 5.474]
 [3.312 2.456 5.573]
 [3.322 2.33  5.41 ]
 [3.405 2.283 5.439]
 [3.241 2.278 5.438]
 [3.321 2.34  5.31 ]
 [3.441 2.551 5.433]
 [3.433 2.66  5.377]
 [3.564 2.505 5.458]
 [3.576 2.417 5.504]
 [3.68  2.583 5.419]
 [3.675 2.596

NOTE) Open file em.tpr to read
NOTE) End of em.tpr to read
